# SpeechLM API in vLLM example

In [ ]:
# Mid-stream context injection demo.
#
# 1. Prefill "My name is"
# 2. Feed "Jensen Huang. " one token at a time (decode_step_shm)
# 3. Inject "I want to present you a new GPU" in one shot (append_request)
# 4. Let the model continue autoregressively

from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM
import torch

engine_args = AsyncEngineArgs(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_model_len=256,
    gpu_memory_utilization=0.8,
    shm_decode=True,
    input_coalesce_timeout_ms=5,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=200, skip_sampling=True)

prompt = "My name is "
prompt_tokens = engine.tokenizer.encode(prompt)
inputs = {
    "prompt_token_ids": [0] * len(prompt_tokens),
    "custom_inputs": {
        "custom_in_tokens": torch.tensor(prompt_tokens, dtype=torch.int32),
    },
}

request_id = "extend_demo_1"

# We track what the model actually saw (the "context" we built),
# separately from what it sampled.
context_tokens = list(prompt_tokens)

# ── 1. Prefill ──
queue = await engine.add_request(request_id, inputs, sampling_params)
prefill_output = await queue.get()
last_token = prefill_output.outputs[0].custom_outputs["custom_out_tokens"][-1].item()
print(f"After prefill: '{engine.tokenizer.decode(context_tokens)}'")
print(f"  (model predicted '{engine.tokenizer.decode([last_token])}' — we override it)")

# ── 2. Feed "Jensen Huang. " one token at a time ──
# We ignore the model's predictions here — we're dictating the text.
# add_special_tokens=False to avoid extra <s> in the middle.
inject_text = "Jensen Huang. "
inject_tokens = engine.tokenizer.encode(inject_text, add_special_tokens=False)
print(f"\nInjecting '{inject_text}' token-by-token ({len(inject_tokens)} tokens)...")

for token in inject_tokens:
    outputs = engine.decode_step_shm(
        request_id,
        custom_inputs={
            "custom_in_tokens": torch.tensor([token], dtype=torch.int32),
        },
    )
    last_token = int(outputs["custom_out_tokens"][0])
    context_tokens.append(token)

print(f"After inject:  '{engine.tokenizer.decode(context_tokens)}'")

# ── 3. Extend with "I want to present you a new GPU" in one shot ──
# Multi-token append_request — all processed in a single forward pass.
extend_text = "I want to present you a new GPU called"
extend_tokens = engine.tokenizer.encode(extend_text, add_special_tokens=False)
print(f"\nExtending with '{extend_text}' ({len(extend_tokens)} tokens, single forward pass)...")

await engine.append_request(
    request_id=request_id,
    custom_inputs={
        "custom_in_tokens": torch.tensor(extend_tokens, dtype=torch.int32),
    },
)

# Output from the re-prefill arrives via ZMQ (not SHM).
extend_output = await queue.get()
last_token = extend_output.outputs[0].custom_outputs["custom_out_tokens"][-1].item()
context_tokens.extend(extend_tokens)
print(f"After extend:  '{engine.tokenizer.decode(context_tokens)}'")

# ── 4. Autoregressive decode — feed model's own predictions back ──
sampled_tokens = [last_token]
print(f"\nDecoding autoregressively...")
for _ in range(30):
    outputs = engine.decode_step_shm(
        request_id,
        custom_inputs={
            "custom_in_tokens": torch.tensor([last_token], dtype=torch.int32),
        },
    )
    last_token = int(outputs["custom_out_tokens"][0])
    sampled_tokens.append(last_token)

await engine.abort(request_id)

print(f"Context:       '{engine.tokenizer.decode(context_tokens)}'")
print(f"Model sampled: '{engine.tokenizer.decode(sampled_tokens)}'")
print(f"Full text:     '{engine.tokenizer.decode(context_tokens + sampled_tokens)}'")